In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Dense,
    Dropout, Flatten, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from Load_and_Preprocess import *

# Load data
x_train, x_test, y_train, y_test, input_shape = load_and_preprocess()

num_classes = 10

# Build CNN Model
model = Sequential([
    
    Conv2D(32, (3, 3), padding='same',
           activation='relu',
           input_shape=input_shape),
    BatchNormalization(),

    Conv2D(32, (3, 3), activation='relu'),
    BatchNormalization(),

    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Conv2D(64, (3, 3), padding='same',
           activation='relu'),
    BatchNormalization(),

    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),

    MaxPooling2D((2, 2)),
    Dropout(0.25),

    Flatten(),

    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(num_classes, activation='softmax')
])

# Compile Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# Train Model
history = model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=20,
    validation_data=(x_test, y_test),
    callbacks=[early_stop, reduce_lr]
)

# Evaluate
score = model.evaluate(x_test, y_test, verbose=0)

print("\nTest Loss:", score[0])
print("Test Accuracy:", score[1])

# Model Summary
model.summary()

In [4]:
# Evaluate model
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

# ==========================================
# Test on image containing multiple digits
# ==========================================

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Load image
img = cv2.imread("digits.jpeg")

# Convert to grayscale
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Threshold
_, thresh = cv2.threshold(
    gray, 128, 255,
    cv2.THRESH_BINARY_INV
)

# Find contours
contours, _ = cv2.findContours(
    thresh,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# Sort contours from left to right
contours = sorted(
    contours,
    key=lambda c: cv2.boundingRect(c)[0]
)

for cnt in contours:

    x, y, w, h = cv2.boundingRect(cnt)

    # Ignore small noise
    if w < 5 or h < 5:
        continue

    # Extract digit
    digit = thresh[y:y+h, x:x+w]

    # Resize to 20x20
    digit = cv2.resize(digit, (20, 20))

    # Create 28x28 image like MNIST
    canvas = np.zeros((28, 28), dtype=np.uint8)
    canvas[4:24, 4:24] = digit

    # Normalize
    test_digit = canvas.astype("float32") / 255.0

    # Reshape according to model input
    if input_shape == (28, 28, 1):
        test_digit = test_digit.reshape(1, 28, 28, 1)
    else:
        test_digit = test_digit.reshape(1, 1, 28, 28)

    # Predict
    prediction = model.predict(test_digit, verbose=0)
    predicted_digit = np.argmax(prediction)

    # Draw result
    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(
        img,
        str(predicted_digit),
        (x, y-10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

# Display result
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Digit Recognition")
plt.axis("off")
plt.show()

NameError: name 'model' is not defined

In [3]:
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

print("Test Accuracy:", accuracy * 100, "%")

predictions = model.predict(x_test, verbose=0)

predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(y_test, axis=1)

correct = np.sum(predicted_labels == true_labels)

print("Correct Predictions :", correct)
print("Total Test Images   :", len(true_labels))
print("Wrong Predictions   :", len(true_labels) - correct)

Test Accuracy: 99.65999722480774 %
Correct Predictions : 9966
Total Test Images   : 10000
Wrong Predictions   : 34


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np

# Predict on test data
y_pred = model.predict(x_test, verbose=0)

# Convert one-hot encoded labels to class labels
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Metrics
accuracy = accuracy_score(y_true, y_pred_classes)
precision = precision_score(y_true, y_pred_classes, average='weighted')
recall = recall_score(y_true, y_pred_classes, average='weighted')
f1 = f1_score(y_true, y_pred_classes, average='weighted')

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

# Detailed report
print("\nClassification Report:")
print(classification_report(y_true, y_pred_classes))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred_classes)
print("\nConfusion Matrix:")
print(cm)

Accuracy : 0.9966
Precision: 0.9966023095203415
Recall   : 0.9966
F1 Score : 0.9965993595977414

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       980
           1       1.00      1.00      1.00      1135
           2       1.00      1.00      1.00      1032
           3       1.00      1.00      1.00      1010
           4       0.99      1.00      1.00       982
           5       1.00      0.99      1.00       892
           6       1.00      1.00      1.00       958
           7       1.00      1.00      1.00      1028
           8       1.00      1.00      1.00       974
           9       1.00      0.99      1.00      1009

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000


Confusion Matrix:
[[ 979    0    0    0    0    0    0    1    0    0]
 [   0 1132    0    1    0    0    2    0    0   

In [2]:
import cv2
import numpy as np

# Create black canvas
canvas = np.zeros((280, 280), dtype=np.uint8)

drawing = False

def draw(event, x, y, flags, param):
    global drawing, canvas

    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True

    elif event == cv2.EVENT_MOUSEMOVE:
        if drawing:
            cv2.circle(canvas, (x, y), 10, 255, -1)

    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False

cv2.namedWindow("Draw Digit")
cv2.setMouseCallback("Draw Digit", draw)

while True:
    cv2.imshow("Draw Digit", canvas)

    key = cv2.waitKey(1) & 0xFF

    # Press P to predict
    if key == ord('p'):

        img = cv2.resize(canvas, (28, 28))
        img = img.astype("float32") / 255.0
        img = img.reshape(1, 28, 28, 1)

        prediction = model.predict(img, verbose=0)
        digit = np.argmax(prediction)
        confidence = np.max(prediction) * 100

        print(f"Predicted Digit: {digit}")
        print(f"Confidence: {confidence:.2f}%")

    # Press C to clear
    elif key == ord('c'):
        canvas[:] = 0

    # Press ESC to exit
    elif key == 27:
        break

cv2.destroyAllWindows()

NameError: name 'model' is not defined